# Phase 1.2: Tier A frozen encoders (tile 32UNU, Allgäu / Upper Swabia)

Five "encoders", one uniform interface:

    encode_bundle(frames: Tensor[T, C, H, W]) -> {"pooled": [T, D],
                                                  "grid": [T, 16, D_grid], ...}

| name | what | D |
|---|---|---|
| `raw_features` | NOT a network: per-band + NDVI summary stats. The row that gives every later probe its meaning | 35 |
| `imagenet_vit_b16` | torchvision ViT-B/16, the non-Earth floor | 1536 |
| `dinov2_vitb14` | torch.hub DINOv2 ViT-B/14, the generalist | 3840 |
| `satlas_s2_swinb_rgb` | SatlasPretrain Swin-B, EO-native, single-image | 1024 |
| `satlas_s2_swinb_mi_rgb` | SatlasPretrain Swin-B MULTI-IMAGE, the positive control | 1024 |

Everything frozen: `.eval()` and `torch.no_grad()`, always. No quality comparison
happens in this phase -- that is Phase 1.3, through `probes/cv.py`, and any number
produced outside `probes/cv.py` does not exist.

Exit test: five wrappers return asserted `[T_kept, D]` on the same cube; the
valid-pixel reflectance assertion passes on all 20 cubes; D printed per model;
this notebook runs clean end to end in a fresh runtime.

Before running: Runtime > Change runtime type > **T4 GPU**, and `phase1_2_repo.zip`
dragged into `My Drive/NeurIPS-CCAI-2026/` (zipped -- see RUNBOOK.md).

## Step 1: Install, then restart

Self-skips on re-run via a sentinel. Exactly one automatic restart, at the end of
this cell; resume at Step 2.

In [ ]:
import os, IPython
SENTINEL = "/content/.phase1_2_installed"

if os.path.exists(SENTINEL):
    print("Already installed in this runtime, skipping.")
    print(f"(delete {SENTINEL} and re-run to force a reinstall)")
else:
    # Not -q. A pip resolution failure here is the likeliest cause of every
    # later failure, and -q hides it.
    !pip install earthnet s3fs xarray zarr netCDF4 satlaspretrain-models

    # Colab ships a CUDA-matched torch. Installing over it swaps in a CPU wheel
    # and makes every encoder far slower, so only act if something is missing.
    import importlib.util
    if (importlib.util.find_spec("torch") is None
            or importlib.util.find_spec("torchvision") is None):
        !pip install torch torchvision

    # Verify before restarting, so a broken install cannot reach the encoders.
    import subprocess, sys
    probe = ("import s3fs, xarray, zarr, netCDF4, earthnet, "
             "torch, torchvision, satlaspretrain_models")
    r = subprocess.run([sys.executable, "-c", probe], capture_output=True, text=True)
    if r.returncode != 0:
        print(r.stdout)
        print(r.stderr)
        raise RuntimeError(
            "Install did not take. Read the pip output above for the real "
            "conflict. Do not continue: Step 6 would fail to build the encoders."
        )

    open(SENTINEL, "w").write("ok")
    print("\n" + "=" * 70)
    print("INSTALL VERIFIED. RESTARTING THE RUNTIME NOW. This is expected.")
    print("When it comes back, continue from Step 2. Do not re-run this cell.")
    print("=" * 70)
    IPython.get_ipython().kernel.do_shutdown(True)

## Step 2: Bootstrap

Mounts Drive, finds and (re)extracts `phase1_2_repo.zip`, puts the repo on
`sys.path`, defines the `sh()` helper every later step uses.

In [ ]:
import os, sys, glob, zipfile, textwrap

REQUIRED = ["data/ndvi.py", "data/loader.py", "data/download_greenearthnet.py",
            "encoders/base.py", "encoders/frames.py", "encoders/pipeline.py",
            "encoders/raw_features.py", "encoders/imagenet_vit.py",
            "encoders/dinov2_vit.py", "encoders/satlas_s2.py",
            "tests/test_encoders.py", "tests/conftest.py"]
ZIP_NAME = "phase1_2_repo.zip"

try:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE = "/content/drive/MyDrive"
except ImportError:
    DRIVE = None
    print("not on Colab, assuming the repo is the current directory")

def looks_like_repo(d):
    return d and all(os.path.exists(os.path.join(d, f)) for f in REQUIRED)

REPO = None
if DRIVE:
    zips = glob.glob(f"{DRIVE}/**/{ZIP_NAME}", recursive=True)
    unzipped = [os.path.dirname(os.path.dirname(h))
                for h in glob.glob(f"{DRIVE}/*/data/ndvi.py")
                + glob.glob(f"{DRIVE}/*/*/data/ndvi.py")]
    unzipped = [d for d in unzipped if looks_like_repo(d)]

    if zips:
        REPO = os.path.dirname(zips[0])
        marker = os.path.join(REPO, "encoders", "base.py")
        # Re-extract when the zip is newer than what is on disk. Without this a
        # freshly uploaded zip is ignored because an old checkout sits next to
        # it, and you debug last week's code.
        stale = (not os.path.exists(marker)
                 or os.path.getmtime(zips[0]) > os.path.getmtime(marker))
        if stale:
            print(f"found {zips[0]}")
            print(f"extracting into {REPO} (zip is newer)")
            with zipfile.ZipFile(zips[0]) as zf:
                zf.extractall(REPO)
        else:
            print(f"using existing checkout at {REPO} (zip is not newer)")
    elif unzipped:
        REPO = unzipped[0]
        print(f"found unzipped repo, no zip present: {REPO}")
else:
    REPO = os.getcwd()

if not looks_like_repo(REPO):
    raise RuntimeError(textwrap.dedent(f"""
        Could not find the Phase 1.2 code.

        Fix, 2 minutes:
          1. Run make_zip.sh locally to build {ZIP_NAME}
          2. Open https://drive.google.com
          3. Drag {ZIP_NAME} into  My Drive / NeurIPS-CCAI-2026  (do not unzip)
          4. Re-run this cell.

        Searched under: {DRIVE}
        Needed all of: {REQUIRED}
        Resolved REPO = {REPO}
    """).strip())

os.chdir(REPO)
if REPO not in sys.path:
    sys.path.insert(0, REPO)
os.environ["PYTHONPATH"] = REPO + os.pathsep + os.environ.get("PYTHONPATH", "")

# Per-phase artefact dirs: re-running one phase never touches another, and
# data/raw stays shared because the cubes are phase-independent.
from data.paths import RAW_DIR, migrate_legacy, phase_dir
PHASE = "phase1_2"
RAW = os.path.join(REPO, RAW_DIR)
os.makedirs(RAW, exist_ok=True)
migrate_legacy()                                   # one-time, idempotent
EMB = phase_dir(PHASE, "embeddings")
MSK = phase_dir(PHASE, "masks")

print(f"\nREPO  {REPO}")
print(f"RAW   {RAW}")
print(f"EMB   {EMB}")
print(f"MSK   {MSK}")
for f in REQUIRED:
    print(f"  ok  {f}")

from data.ndvi import ndvi
from data.loader import S2_BANDS
from encoders import TIER_A, build_encoder
print(f"\nimports OK. canonical NDVI at {ndvi.__module__}, bands {S2_BANDS}")
print(f"Tier A: {TIER_A}")


# --- shell helper, defined here so it can never be skipped ------------------
# Named sh(), not run(): IPython has a %run magic. If a helper called run() is
# ever undefined, automagic silently rewrites run("...") into %run("...") and
# reports a confusing error about a missing script instead of a NameError.
#
# PYTHONUNBUFFERED matters: a subprocess writing to a pipe block-buffers stdout,
# so progress lines sit invisible for minutes then arrive in one burst. That is
# what makes a working download look like a hang.
import subprocess

def sh(cmd, cwd=None):
    print("$", cmd, flush=True)
    proc = subprocess.Popen(cmd, shell=True, cwd=cwd or REPO, text=True, bufsize=1,
                            stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                            env={**os.environ, "PYTHONUNBUFFERED": "1"})
    for line in proc.stdout:
        print(line, end="")
    if proc.wait() != 0:
        raise RuntimeError(f"command failed with exit code {proc.returncode}: {cmd}")
    print(f"[exit 0] {cmd}")

print("helper ready: sh('<shell command>')")

## Step 3: Environment check

In [ ]:
import glob, os, shutil
import torch, torchvision, numpy as np
import satlaspretrain_models

print("torch      ", torch.__version__, "| cuda:", torch.cuda.is_available(),
      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU ONLY")
print("torchvision", torchvision.__version__)
print("satlaspretrain_models imported OK "
      f"({os.path.dirname(satlaspretrain_models.__file__)})")

if torch.cuda.is_available():
    print(f"GPU mem     {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB"
          "  (budget for this block: T4, <= 12 GB)")
else:
    print("no GPU. The encoders run on CPU too -- slower, identical numbers.")

n_cubes = len(glob.glob(os.path.join(RAW, "*.nc")))
print(f"cubes      {n_cubes} in {RAW}"
      + ("" if n_cubes >= 20 else "   (Step 5 downloads the rest)"))
free = shutil.disk_usage(RAW).free / 1e9
print(f"free space {free:.1f} GB on Drive (embeddings need ~0.01 GB; "
      "model weights ~1.5 GB go to the LOCAL runtime disk, not Drive)")
assert free > 1, "less than 1 GB free, clear space on Drive first" 

## Step 4: Unit tests

Expect **`111 passed, 5 skipped`**. The 5 skips are the wrapper tests that would
download pretrained weights inside pytest (`PHASE1_2_WEIGHTS=1` enables them);
Steps 6-11 exercise those same wrappers for real on the actual cubes, which is
the stronger test anyway.

In [ ]:
sh("python -m pytest tests -q")

## Step 5: The cubes

Idempotent: cubes already on Drive are skipped. 15 s from scratch.

In [ ]:
sh(f"python -m data.download_greenearthnet --out '{RAW}' --n 20 --tile 32UNU")

from data.loader import assert_no_overlap
paths = assert_no_overlap(RAW)
print(f"{len(paths)} cubes on disk, no two share a pixel")

## Step 6: Build the five encoders

First run downloads weights (torchvision ViT ~330 MB, DINOv2 ~330 MB via
torch.hub, Satlas Swin-B ~200 MB) to the local runtime disk. Every wrapper
prints its **explicit preprocessing** -- read it; nothing radiometric happens
implicitly, and the raw baseline prints why it exists.

In [ ]:
import torch
from encoders import TIER_A, build_encoder

device = "cuda" if torch.cuda.is_available() else "cpu"
ENCODERS = {}
for name in TIER_A:
    print("=" * 70)
    ENCODERS[name] = build_encoder(name, device=device)

print("=" * 70)
print("D per model:")
for name, enc in ENCODERS.items():
    print(f"  {name:22s} D={enc.embed_dim}")

## Step 7: Five wrappers, one cube, asserted `[T_kept, D]`

The retained-frame count is cross-checked against an independent computation
that never touches the `encoders` package: plain numpy on the loader's mask.

That independent count applies the same **finite** rule the pipeline does. A
pixel can be both mask-valid and no-data -- GreenEarthNet's clear-sky
conjunction reads the mask bands and never looks at the reflectance bands, so
113 such pixels exist across these 20 cubes. A pixel carrying no reflectance
value is not an observation, so it does not count towards a frame being clear.
This is the rule `data.ndvi.ndvi` already applies internally; nothing is
filled, pixels only ever leave the valid set. On this tile it changes no
frame's keep/drop decision.

In [ ]:
import numpy as np, pandas as pd
from data.loader import iter_cubes
from encoders.pipeline import encode_cube

sample = next(iter_cubes(RAW, limit=1))
T, C, H, W = sample.values.shape

# Independent retained-frame count (nothing from encoders/ involved).
corrected = sample.mask & np.isfinite(sample.values).all(axis=1)
indep = int(sum(np.count_nonzero(corrected[t]) / (H * W) > 0.5 for t in range(T)))
print(f"independent count of frames with clear-fraction > 0.5: {indep}/{T}")
print(f"mask-valid but no-data pixels demoted: "
      f"{int((sample.mask & ~np.isfinite(sample.values).all(axis=1)).sum())}\n")

encoded, rows = {}, []
for name, enc in ENCODERS.items():
    print("=" * 70)
    ec = encode_cube(sample, enc)
    assert ec.embeddings.shape == (indep, enc.embed_dim), (
        f"{name}: {ec.embeddings.shape} != ({indep}, {enc.embed_dim})")
    encoded[name] = ec
    rows.append({"encoder": name, "D": enc.embed_dim, "T_kept": ec.embeddings.shape[0],
                 "clear_min": round(float(ec.clear_frac.min()), 3),
                 "clear_median": round(float(np.median(ec.clear_frac)), 3),
                 "clear_max": round(float(ec.clear_frac.max()), 3)})

print("=" * 70)
ts0 = encoded["raw_features"].timestamps
same = all(np.array_equal(ec.timestamps, ts0) for ec in encoded.values())
print(f"kept frames identical across all five encoders: {same}")
assert same
pd.DataFrame(rows)

## Step 8: Valid-pixel reflectance, all 20 cubes

The global max may hit ~1.98 (bright cloud) -- harmless **only** because those
pixels are masked. Reflectance above 1.2 at a pixel the mask calls *valid* is
not physically plausible for a surface, so it is counted here.

The check asserts on **prevalence, not on the maximum**. A maximum over ~17
million pixels is the most outlier-sensitive statistic there is, and tile 32UNU
contains 44 isolated valid pixels above 1.2 (2.5e-6) -- specular targets in
99.7-100% clear frames that move no downstream number by more than 4e-4
relative. The failure worth halting for is *cloud passing as clear over real
area*, which would land near 1e-1, about a thousand times the 1e-4 tolerance.
Expect every cube to print a prevalence at or below ~1.9e-5.

In [ ]:
import os
from data.loader import iter_cubes
from encoders.frames import assert_valid_reflectance, MAX_IMPLAUSIBLE_FRACTION

n, worst = 0, 0.0
for s in iter_cubes(RAW, limit=20, verbose=False):
    rep = assert_valid_reflectance(s.values, s.mask, verbose=False)
    worst = max(worst, rep.fraction)
    print(f"  ok  valid_max={rep.valid_max:6.4f} | all-finite max={rep.global_max:6.4f} "
          f"| implausible {rep.n_implausible:3d} ({rep.fraction:.2e})  "
          f"{os.path.basename(s.path)[:44]}")
    n += 1
assert n == 20, f"expected 20 cubes, checked {n}"
print(f"\nall {n} cubes pass: worst prevalence {worst:.2e}, "
      f"tolerance {MAX_IMPLAUSIBLE_FRACTION:.0e}")
print("bright cloud stays behind the mask; what is left is isolated and counted")

## Step 9: Wrong-shaped input fails loudly

Every wrapper must refuse a malformed batch with a readable assertion --
never a silent broadcast into a wrong-but-plausible embedding.

In [ ]:
import torch

bad_inputs = [
    ("rank 3, no band axis  (T, H, W)", torch.rand(5, 128, 128)),
    ("channels-last  (T, H, W, C)", torch.rand(5, 128, 128, 4)),
    ("empty batch  (0, C, H, W)", torch.rand(0, 4, 128, 128)),
]
for name, enc in ENCODERS.items():
    print(f"[{name}]")
    for desc, bad in bad_inputs:
        try:
            enc.encode(bad, verbose=False)
        except AssertionError as e:
            print(f"  refused {desc}: {str(e).splitlines()[0][:88]}")
        else:
            raise RuntimeError(
                f"{name} ACCEPTED {desc}: silent broadcast, fix the wrapper")

# The baseline must also refuse to run without the mask the canonical NDVI needs.
try:
    ENCODERS["raw_features"].encode(torch.rand(2, 4, 128, 128) * 0.4, verbose=False)
except AssertionError as e:
    print(f"[raw_features]\n  refused missing mask: {str(e).splitlines()[0][:88]}")
else:
    raise RuntimeError("raw_features ran without the mask that data.ndvi.ndvi requires")

print("\nall malformed inputs refused loudly")

## Step 10: Memory does not scale with T

The seasonal split has ~290 frames per cube, ten times this subset. Random
frames through a real model at `batch_size=16`: peak GPU memory must stay far
under the 12 GB budget, and the only thing that grows with T is time.

In [ ]:
import torch

T290 = 290
big = torch.rand(T290, 4, 128, 128) * 0.4   # ~76 MB host-side; random by design
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()

z = ENCODERS["imagenet_vit_b16"].encode(big, batch_size=16)
assert z.shape == (T290, ENCODERS["imagenet_vit_b16"].embed_dim)

if torch.cuda.is_available():
    peak = torch.cuda.max_memory_allocated() / 1e9
    print(f"peak GPU memory {peak:.2f} GB for T={T290} at batch_size=16")
    assert peak < 12, f"{peak:.2f} GB blows the T4 budget"
del big, z

## Step 11: Encode all 20 cubes with all five, save alongside clear-fractions

Each `.npz` carries the embeddings `[T_kept, D]`, the surviving timestamps,
their **exact clear-fractions** (so later probes can filter more strictly
without re-encoding), and their indices into the original cube time axis.

Resumable: existing files are loaded and re-asserted, not re-encoded. Delete
`data/embeddings/` to force a full re-encode (do that after any change to the
mask definition or the frame-selection rule).

In [ ]:
import os
import numpy as np, pandas as pd
from data.loader import iter_cubes
from encoders.pipeline import (encode_cube, save_encoded, load_encoded,
                               cube_masks, save_masks, SCHEMA_VERSION)

print(f"cache schema v{SCHEMA_VERSION}. A file from an older schema is REFUSED "
      "on load, not silently reused: it would be missing fields (window_span_days
"
      "is the live example) that probes read as absent. To force a clean re-encode:
"
      "    from data.paths import reset_phase; reset_phase('phase1_2')
")

rows = []
for s in iter_cubes(RAW, limit=20, verbose=False):
    cube = os.path.basename(s.path)
    save_masks(MSK, cube_masks(s, verbose=False), verbose=False)
    for name, enc in ENCODERS.items():
        path = os.path.join(EMB, f"{os.path.splitext(cube)[0]}__{name}.npz")
        if os.path.exists(path):
            ec = load_encoded(path)                  # re-asserts every invariant
            status = "cached"
        else:
            ec = encode_cube(s, enc, verbose=False)
            save_encoded(EMB, ec, verbose=False)
            status = "encoded"
        assert ec.embeddings.shape == (ec.clear_frac.shape[0], enc.embed_dim)
        rows.append({"cube": cube, "encoder": name, "D": enc.embed_dim,
                     "T_kept": ec.embeddings.shape[0],
                     "clear_median": float(np.median(ec.clear_frac)),
                     "status": status})
        print(f"  {status:7s} {name:22s} [{ec.embeddings.shape[0]:2d}, {enc.embed_dim:4d}]  {cube}")

df = pd.DataFrame(rows)
assert (df.groupby("encoder").cube.nunique() == 20).all(), "an encoder missed a cube"
print(f"\n{len(df)} cube x encoder pairs in {EMB}")
(df.groupby("encoder", sort=False)
   .agg(cubes=("cube", "nunique"), D=("D", "first"),
        frames_total=("T_kept", "sum"),
        T_kept_min=("T_kept", "min"), T_kept_median=("T_kept", "median")))

## Phase 1.2 is done when

- Step 6 printed D for all five models: **35, 1536, 3840, 1024, 1024**
- Step 7 returned four asserted `[T_kept, D]` on the same cube, with `T_kept`
  equal to the independent count and identical timestamps across encoders
  (expect `T_kept` in 10-16, matching `log.md`'s Phase 1.1 clear-frame counts)
- Step 8 passed the valid-pixel reflectance assertion on **all 20 cubes**,
  worst prevalence ~1.9e-5 against a 1e-4 tolerance
- Step 9 saw every wrapper refuse every malformed input loudly
- Step 11 left **100** `.npz` files in `data/embeddings/`, each carrying exact
  per-frame clear-fractions

Then:

- **Phase 1.3: `probes/cv.py` fold generators.** Until they exist, nothing here
  is a result -- these embeddings are inputs, not numbers.
- Probes filter frames by the stored `clear_frac` (e.g. a stricter 0.8 cut)
  without re-encoding anything.